# AMPS II · Week 07 — Lab
## Trusses, 3D Timoshenko frames and the geometrically exact rod

| | |
|---|---|
| **Lecture sections** | Part A (trusses on 1D meshes) · Part B (Timoshenko frames, locking) · Part C (finite rotations) · Part D (LLM) |
| **Time** | ≈ 80 min in class |
| **You will** | solve a truss and check it against the method of joints · see shear locking and its cure · load a frame in bending and torsion · roll a rod into a circle and a helix · compare a large-deflection cantilever with the elastica |

**Reminder.** `%%fenicsx` cells are separate MPI programs; ordinary cells have NumPy and matplotlib only. Trusses and frames (joints where 3 or more members
meet) run with `-np 1` — §2 shows why. The two model modules are written once with `%%writefile` (§1).

## ⚙️ Setup — run once per runtime
Installs DOLFINx 0.11 and registers the `%%fenicsx` cell magic. Re-run after every runtime restart.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  FEniCSx on Google Colab — run once per runtime (≈ 3–5 min)
#  https://github.com/SeoulTechPSE/fenicsx-colab
# ══════════════════════════════════════════════════════════════════
from google.colab import drive
import os, multiprocessing, subprocess
from pathlib import Path

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/seoultechpse/fenicsx-colab.git'
REPO_DIR = Path('/content/fenicsx-colab')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

# OpenMPI 5.x slot patch: Colab has 2 CPUs; allow up to 4 ranks (oversubscribed)
N_PROC = 4
with open('/root/hostfile', 'w') as f:
    f.write(f'localhost slots={N_PROC}\n')
os.environ['OMPI_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['PRTE_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['OMPI_MCA_osc'] = '^ucx'   # skip the UCX one-sided module (harmless start-up message)

DOLFINX_VERSION = '0.11'
USE_COMPLEX = False
USE_CLEAN   = False
opts = [f'--version {DOLFINX_VERSION}']
if USE_COMPLEX: opts.append('--complex')
if USE_CLEAN:   opts.append('--clean')
get_ipython().run_line_magic('run', f"{REPO_DIR / 'setup_fenicsx.py'} {' '.join(opts)}")

# Folder shared by %%fenicsx cells (computation) and ordinary cells (plotting)
os.makedirs('/tmp/fnx', exist_ok=True)

In [ ]:
%%fenicsx --info -np 2

## 1 · The model modules  *(lecture §A1–A3, §B1–B4, §C1–C4)*
`frame.py`: a 1D mesh from joints and members (`mesh.create_mesh`, no Gmsh), `Truss` (axial strain $\boldsymbol t\cdot\nabla\mathbf u\,\boldsymbol t$ from the cell Jacobian, member forces), `Frame`
(3D Timoshenko: $(\mathbf u,\boldsymbol\theta)$, local frame $(\boldsymbol t,\mathbf a_1,\mathbf a_2)$, element choice `P1` / `P1-RI` / `P2P1`, section forces), and the section helpers `rect`, `circ`.
`rod.py`: the geometrically exact rod (rotation vector, $\Lambda=\exp\widehat{\boldsymbol\theta}$, energy → `ufl.derivative` → SNES) and three exact references: the circle, the helix, the
(extensible) elastica. Run the two cells.

In [ ]:
%%writefile /tmp/fnx/frame.py
"""Linear trusses and 3D Timoshenko frames on 1D meshes embedded in 2D/3D, DOLFINx 0.11.
Members are straight lines between joints; each member is cut into n cells; the mesh is built with mesh.create_mesh (no Gmsh).
SERIAL ONLY when 3 or more members meet at a joint: in parallel (0.11) such a vertex is duplicated when its cells lie on
different ranks, and the structure falls apart. A chain (every vertex in at most 2 cells) is fine in parallel."""
import numpy as np
import ufl, basix.ufl
from mpi4py import MPI
from petsc4py import PETSc
from dolfinx import fem, mesh
from dolfinx.fem.petsc import assemble_matrix, assemble_vector, apply_lifting, set_bc

comm = MPI.COMM_WORLD
def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

def lu(ksp):
    ksp.setType("preonly"); ksp.getPC().setType("lu"); ksp.getPC().setFactorSolverType("mumps")
    return ksp

def line_mesh(X, members, n=1):
    """joints X (k x gdim), members = pairs of joint indices, each member cut into n cells.
    Returns the mesh and the member of every (local) cell."""
    X = np.asarray(X, dtype=float); gdim = X.shape[1]
    pts, cells, owner = [p for p in X], [], []
    for m, (i, j) in enumerate(members):
        prev = i
        for k in range(1, n + 1):
            if k < n:
                pts.append(X[i] + (X[j] - X[i]) * k / n); cur = len(pts) - 1
            else:
                cur = j
            cells.append((prev, cur)); owner.append(m); prev = cur
    if comm.rank == 0:
        x, c = np.array(pts), np.array(cells, dtype=np.int64)
    else:
        x, c = np.zeros((0, gdim)), np.zeros((0, 2), dtype=np.int64)
    msh = mesh.create_mesh(comm, c, ufl.Mesh(basix.ufl.element("Lagrange", "interval", 1, shape=(gdim,))), x)
    msh.topology.create_connectivity(0, 1)                     # needed by locate_dofs_topological on vertices
    return msh, np.array(owner, dtype=int)[msh.topology.original_cell_index]

def tangent(msh):
    """unit tangent of the cells, from the Jacobian of the 1D cell (its sign depends on the cell orientation: use it squared)"""
    J = ufl.Jacobian(msh)
    t = ufl.as_vector([J[i, 0] for i in range(msh.geometry.dim)])
    return t / ufl.sqrt(ufl.dot(t, t))

def vertex_dofs(V, x0, tol=1e-9):
    """DOFs of the (sub)space V at the joint x0. V may be a pair (V.sub(i), collapsed) as for dirichletbc with a Function."""
    msh = (V[0] if isinstance(V, tuple) else V).mesh
    x0 = np.asarray(x0, dtype=float)
    vs = mesh.locate_entities(msh, 0, lambda p: np.all(np.abs(p[:len(x0)].T - x0) < tol, axis=1))
    return fem.locate_dofs_topological(V, 0, vs)

def _add_point_values(b, V, x0, vals, nown):
    for c, f in enumerate(vals):
        if f != 0.0:
            for d in vertex_dofs(V.sub(c), x0):
                if d < nown:
                    b.array[d] += f


class Truss:
    """pin-jointed truss, u in P1. One cell per member: a member cut into more cells leaves the transverse motion of its inner
    nodes without stiffness (singular matrix). ES: one value or one per member; eps0: initial (lack-of-fit / thermal) strain
    per member, N = ES (eps - eps0)."""
    def __init__(self, X, members, ES, n=1, eps0=0.0):
        self.msh, self.member = line_mesh(X, members, n)
        msh = self.msh; gdim = self.gdim = msh.geometry.dim; nm = len(members)
        V = self.V = fem.functionspace(msh, ("Lagrange", 1, (gdim,)))
        t = tangent(msh)
        self.eps = lambda w: ufl.dot(ufl.dot(ufl.grad(w), t), t)
        Q = fem.functionspace(msh, ("DG", 0))
        self.ES, self.eps0 = fem.Function(Q), fem.Function(Q)
        self.ES.x.array[:] = np.broadcast_to(np.asarray(ES, dtype=float), (nm,))[self.member]
        self.eps0.x.array[:] = np.broadcast_to(np.asarray(eps0, dtype=float), (nm,))[self.member]
        u, v = ufl.TrialFunction(V), ufl.TestFunction(V)
        dx = ufl.Measure("dx", domain=msh)
        self.a = fem.form(self.ES * self.eps(u) * self.eps(v) * dx)
        self.L = fem.form(self.ES * self.eps0 * self.eps(v) * dx)
        self.bcs, self.loads = [], []
        self.u = fem.Function(V)

    def fix(self, x0, comps=None):
        """fix the components comps (None = all) of the joint at x0"""
        if comps is None:
            self.bcs.append(fem.dirichletbc(np.zeros(self.gdim), vertex_dofs(self.V, x0), self.V))
        else:
            for c in comps:
                self.bcs.append(fem.dirichletbc(0.0, vertex_dofs(self.V.sub(c), x0), self.V.sub(c)))

    def load(self, x0, F):
        self.loads.append((np.asarray(x0, dtype=float), np.asarray(F, dtype=float)))

    def solve(self):
        A = assemble_matrix(self.a, bcs=self.bcs); A.assemble()
        b = assemble_vector(self.L)
        nown = self.V.dofmap.index_map.size_local * self.gdim
        for x0, F in self.loads:
            _add_point_values(b, self.V, x0, F, nown)
        apply_lifting(b, [self.a], [self.bcs]); b.ghostUpdate(PETSc.InsertMode.ADD, PETSc.ScatterMode.REVERSE); set_bc(b, self.bcs)
        k = lu(PETSc.KSP().create(comm)); k.setOperators(A); k.solve(b, self.u.x.petsc_vec); self.u.x.scatter_forward()
        return k.getConvergedReason()

    def forces(self):
        """normal force N = ES (eps - eps0) of every member, in member order (serial)"""
        Q = self.ES.function_space; N = fem.Function(Q)
        N.interpolate(fem.Expression(self.ES * (self.eps(self.u) - self.eps0), Q.element.interpolation_points))
        out = np.zeros(self.member.max() + 1)
        out[self.member] = N.x.array[:len(self.member)]
        return out


class Frame:
    """3D Timoshenko frame, unknowns (u, theta). Local frame per member: a1 = up x t / |up x t|, a2 = t x a1, with a reference
    vector 'up' (one vector, or one per member) that must NOT be parallel to the member. Section dict: ES, kGS1, kGS2, EI1, EI2, GJ,
    with I1 = int x2^2 dA (bending about a1 = deflection along a2) and I2 = int x1^2 dA.
    elem = "P1" (P1/P1, full integration: shear locking), "P1-RI" (one-point integration of the shear terms), "P2P1" (u in P2)."""
    def __init__(self, X, members, sec, n=10, elem="P1-RI", up=(0.0, 0.0, 1.0)):
        self.msh, self.member = line_mesh(X, members, n)
        msh = self.msh; cell = msh.basix_cell()
        V = self.V = fem.functionspace(msh, basix.ufl.mixed_element([
            basix.ufl.element("Lagrange", cell, 2 if elem == "P2P1" else 1, shape=(3,)), basix.ufl.element("Lagrange", cell, 1, shape=(3,))]))
        t = self.t = tangent(msh)
        up = np.asarray(up, dtype=float)
        if up.ndim == 2:                                       # one reference vector per member: a DG0 vector field
            U = fem.Function(fem.functionspace(msh, ("DG", 0, (3,))))
            U.x.array[:] = up[self.member].ravel(); upv = U
        else:
            upv = ufl.as_vector(up)
        a1 = ufl.cross(upv, t); a1 = a1 / ufl.sqrt(ufl.dot(a1, a1)); a2 = ufl.cross(t, a1)
        self.a1, self.a2 = a1, a2
        s = self.sec = {k: fem.Constant(msh, float(v)) for k, v in sec.items()}
        tg = lambda w: ufl.dot(ufl.grad(w), t)                  # d/ds along the member
        def strains(u, th):
            return dict(N=ufl.dot(tg(u), t), T=ufl.dot(tg(th), t), M1=ufl.dot(tg(th), a1), M2=ufl.dot(tg(th), a2),
                        Q1=ufl.dot(tg(u), a1) - ufl.dot(th, a2), Q2=ufl.dot(tg(u), a2) + ufl.dot(th, a1))
        self.strains = strains
        self.stiff = dict(N="ES", T="GJ", M1="EI1", M2="EI2", Q1="kGS1", Q2="kGS2")
        dx = ufl.Measure("dx", domain=msh)
        dxs = ufl.Measure("dx", domain=msh, metadata={"quadrature_degree": 0}) if elem == "P1-RI" else dx
        (u, th), (v, ph) = ufl.TrialFunctions(V), ufl.TestFunctions(V)
        e, ev = strains(u, th), strains(v, ph)
        self.a = fem.form(sum(s[self.stiff[k]] * e[k] * ev[k] for k in ("N", "T", "M1", "M2")) * dx
                          + sum(s[self.stiff[k]] * e[k] * ev[k] for k in ("Q1", "Q2")) * dxs)
        self.bcs, self.loads = [], []
        self.w = fem.Function(V)

    def clamp(self, x0):
        for i in (0, 1):
            Vc, _ = self.V.sub(i).collapse()
            self.bcs.append(fem.dirichletbc(fem.Function(Vc), vertex_dofs((self.V.sub(i), Vc), x0), self.V.sub(i)))

    def load(self, x0, F=(0, 0, 0), M=(0, 0, 0)):
        self.loads.append((np.asarray(x0, dtype=float), np.asarray(F, dtype=float), np.asarray(M, dtype=float)))

    def solve(self):
        A = assemble_matrix(self.a, bcs=self.bcs); A.assemble()
        b = self.w.x.petsc_vec.duplicate(); b.set(0.0)
        nown = self.V.dofmap.index_map.size_local * self.V.dofmap.index_map_bs
        for x0, F, M in self.loads:
            _add_point_values(b, self.V.sub(0), x0, F, nown); _add_point_values(b, self.V.sub(1), x0, M, nown)
        apply_lifting(b, [self.a], [self.bcs]); b.ghostUpdate(PETSc.InsertMode.ADD, PETSc.ScatterMode.REVERSE); set_bc(b, self.bcs)
        k = lu(PETSc.KSP().create(comm)); k.setOperators(A); k.solve(b, self.w.x.petsc_vec); self.w.x.scatter_forward()
        return k.getConvergedReason()

    def at(self, x0):
        """(u, theta) at the joint x0 (serial)"""
        return tuple(np.array([self.w.x.array[vertex_dofs(self.V.sub(i).sub(c), x0)[0]] for c in range(3)]) for i in (0, 1))

    def resultants(self):
        """section forces per cell (DG0 at the cell midpoint): dict N, Q1, Q2, T, M1, M2 and the midpoints 'x' (serial)"""
        Q = fem.functionspace(self.msh, ("DG", 0)); u, th = self.w.split(); e = self.strains(u, th); out = {}
        for k in e:
            f = fem.Function(Q); f.interpolate(fem.Expression(self.sec[self.stiff[k]] * e[k], Q.element.interpolation_points))
            out[k] = f.x.array.copy()
        X = fem.Function(fem.functionspace(self.msh, ("DG", 0, (3,))))
        X.interpolate(fem.Expression(ufl.SpatialCoordinate(self.msh), X.function_space.element.interpolation_points))
        out["x"] = X.x.array.reshape(-1, 3).copy()
        return out

def rect(b, h, E, nu, kappa=5 / 6):
    """rectangle, b along a1 and h along a2: I1 = b h^3/12 (deflection along a2), I2 = h b^3/12; St-Venant J from the series"""
    G = E / (2 * (1 + nu)); A = b * h; p, q = max(b, h), min(b, h)
    J = p * q**3 * (1 / 3 - 64 / np.pi**5 * q / p * sum(np.tanh((2 * k + 1) * np.pi * p / (2 * q)) / (2 * k + 1)**5 for k in range(50)))
    return dict(ES=E * A, kGS1=kappa * G * A, kGS2=kappa * G * A, EI1=E * b * h**3 / 12, EI2=E * h * b**3 / 12, GJ=G * J)

def circ(r, E, nu):
    """solid circle: I = pi r^4/4, J = pi r^4/2, kappa = 6 (1 + nu)/(7 + 6 nu) (Cowper)"""
    G = E / (2 * (1 + nu)); A = np.pi * r**2; I = np.pi * r**4 / 4; k = 6 * (1 + nu) / (7 + 6 * nu)
    return dict(ES=E * A, kGS1=k * G * A, kGS2=k * G * A, EI1=E * I, EI2=E * I, GJ=G * 2 * I)

In [ ]:
%%writefile /tmp/fnx/rod.py
"""Geometrically exact (Simo-Reissner) rod in 3D, DOLFINx 0.11. Total formulation with the rotation vector theta:
Lambda = exp(Skew theta) (Rodrigues),  Gamma = Lambda^T (t0 + u') - t0  (axial and shear strain),  K = T(theta)^T theta'
(curvature and twist; T = the tangent operator of the exponential map),  energy 1/2 Gamma.C_N Gamma + 1/2 K.C_M K,
residual = ufl.derivative of the potential, Newton = SNES. Straight rod on the x axis, clamped at x = 0, dead tip force F and
a tip moment M with the work M . theta(L) (exact for a rotation about a fixed axis, as in all examples here).
elem: "P1" (u, theta in P1, full integration: locks), "P1-RI" (one-point rule for the force term), "P2P1" (u in P2, theta in P1,
full integration), "P2P1-RI" (two-point rule for the force term)."""
import numpy as np
import ufl, basix.ufl
from mpi4py import MPI
from dolfinx import fem, mesh
from dolfinx.fem.petsc import NonlinearProblem

comm = MPI.COMM_WORLD
def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

def skew(n):
    return ufl.as_matrix([[0, -n[2], n[1]], [n[2], 0, -n[0]], [-n[1], n[0], 0]])

def expmap(th, eps=1e-12):
    """Lambda and T of the rotation vector th (|th| -> 0 regularised by eps)"""
    x = ufl.sqrt(ufl.dot(th, th) + eps); S = skew(th); I = ufl.Identity(3)
    Lam = I + ufl.sin(x) / x * S + (1 - ufl.cos(x)) / x**2 * S * S
    T = I + (1 - ufl.cos(x)) / x**2 * S + (x - ufl.sin(x)) / x**3 * S * S
    return Lam, T

def rodrigues(th):
    """NumPy: the rotation matrix of a rotation vector"""
    th = np.asarray(th, dtype=float); x = np.linalg.norm(th)
    if x < 1e-14:
        return np.eye(3)
    S = np.array([[0, -th[2], th[1]], [th[2], 0, -th[0]], [-th[1], th[0], 0]])
    return np.eye(3) + np.sin(x) / x * S + (1 - np.cos(x)) / x**2 * S @ S

def rod_mesh(L, n):
    """n cells on [0, L] x {0} x {0}: a 1D mesh in 3D (u and theta get 3 components). A chain: parallel is fine."""
    if comm.rank == 0:
        x = np.c_[np.linspace(0, L, n + 1), np.zeros(n + 1), np.zeros(n + 1)]
        c = np.c_[np.arange(n), np.arange(1, n + 1)].astype(np.int64)
    else:
        x, c = np.zeros((0, 3)), np.zeros((0, 2), dtype=np.int64)
    return mesh.create_mesh(comm, c, ufl.Mesh(basix.ufl.element("Lagrange", "interval", 1, shape=(3,))), x)

class Rod:
    """C_N = diag(ES, GS, GS), C_M = diag(GJ, EI, EI) in the material frame (e1 = the rod axis)."""
    count = 0                       # a new options prefix per rod: PETSc keeps options of a prefix for every later solver using it
    def __init__(self, L, n, ES, GS, GJ, EI, elem="P2P1-RI", linesearch="basic", opts=None):
        self.L = L
        msh = self.msh = rod_mesh(L, n); cell = msh.basix_cell()
        V = self.V = fem.functionspace(msh, basix.ufl.mixed_element([
            basix.ufl.element("Lagrange", cell, 2 if elem.startswith("P2") else 1, shape=(3,)), basix.ufl.element("Lagrange", cell, 1, shape=(3,))]))
        self.w = fem.Function(V); u, th = ufl.split(self.w)
        t0 = ufl.as_vector([1.0, 0.0, 0.0])
        Lam, T = expmap(th)
        self.Gam = Lam.T * (t0 + u.dx(0)) - t0                  # the mesh is the x axis: d/ds = d/dx
        self.Kap = T.T * th.dx(0)
        CN, CM = ufl.diag(ufl.as_vector([ES, GS, GS])), ufl.diag(ufl.as_vector([GJ, EI, EI]))
        qN = {"P1": 4, "P1-RI": 1, "P2P1": 4, "P2P1-RI": 2}[elem]        # points of the rule for the force term: 1 (deg 1), 2 (deg 2)
        dxN = ufl.Measure("dx", domain=msh, metadata={"quadrature_degree": qN})
        dx = ufl.Measure("dx", domain=msh, metadata={"quadrature_degree": 4})
        tip = mesh.locate_entities_boundary(msh, 0, lambda x: np.isclose(x[0], L))
        ds = ufl.Measure("ds", domain=msh, subdomain_data=mesh.meshtags(msh, 0, tip, np.ones(len(tip), dtype=np.int32)))
        self.F, self.M = fem.Constant(msh, np.zeros(3)), fem.Constant(msh, np.zeros(3))
        self.Pi_int = 0.5 * ufl.dot(self.Gam, CN * self.Gam) * dxN + 0.5 * ufl.dot(self.Kap, CM * self.Kap) * dx
        Pi = self.Pi_int - ufl.dot(self.F, u) * ds(1) - ufl.dot(self.M, th) * ds(1)
        root = mesh.locate_entities_boundary(msh, 0, lambda x: np.isclose(x[0], 0.0))
        self.bcs = []
        for i in (0, 1):
            Vc, _ = V.sub(i).collapse()
            self.bcs.append(fem.dirichletbc(fem.Function(Vc), fem.locate_dofs_topological((V.sub(i), Vc), 0, root), V.sub(i)))
        o = {"snes_type": "newtonls", "snes_linesearch_type": linesearch, "snes_rtol": 1e-10, "snes_atol": 1e-12, "snes_max_it": 50,
             "snes_divergence_tolerance": 1e12,      # the first Newton update of a stiff rod is the LINEAR solution: the residual grows
             "ksp_type": "preonly", "pc_type": "lu", "pc_factor_mat_solver_type": "mumps"}     # x 1e4 before Newton converges
        o.update(opts or {})
        Rod.count += 1
        self.problem = NonlinearProblem(ufl.derivative(Pi, self.w, ufl.TestFunction(V)), self.w, bcs=self.bcs,
                                        petsc_options_prefix=f"rod{Rod.count}_", petsc_options=o)
        self.tip_forms = [fem.form(self.w.sub(i)[c] * ds(1)) for i in (0, 1) for c in range(3)]
        self.energy_form = fem.form(self.Pi_int)

    def solve(self, F=None, M=None):
        """set the TOTAL tip loads and solve from the current state; returns (SNES reason, Newton iterations)"""
        if F is not None:
            self.F.value[:] = F
        if M is not None:
            self.M.value[:] = M
        self.problem.solve()
        s = self.problem.solver
        return s.getConvergedReason(), s.getIterationNumber()

    def tip(self):
        """(u, theta) at x = L"""
        v = np.array([comm.allreduce(fem.assemble_scalar(f), op=MPI.SUM) for f in self.tip_forms])
        return v[:3], v[3:]

    def energy(self):
        return comm.allreduce(fem.assemble_scalar(self.energy_form), op=MPI.SUM)

    def centerline(self):
        """deformed centre line x + u at the P1 vertices, sorted by x (serial)"""
        Vu, m = self.V.sub(0).collapse(); X = Vu.tabulate_dof_coordinates()
        u = self.w.x.array[m].reshape(-1, 3); i = np.argsort(X[:, 0])
        return X[i] + u[i]

def circle_tip(L, EI, M):
    """pure end moment M about -y: a circular arc of curvature M/EI (any ES, GS: no force, no strain): tip displacement"""
    p = M * L / EI
    return np.array([L * np.sin(p) / p - L, 0.0, L * (1 - np.cos(p)) / p])

def helix_tip(L, EI, m):
    """isotropic rod (EI = GJ) under a dead end moment m: k = m/EI is constant in space, theta(s) = s k, the centre line is a
    helix about m. Tip displacement int_0^L Lambda(s k) e1 ds - L e1 (closed form)."""
    k = np.asarray(m, dtype=float) / EI; a = np.linalg.norm(k); n = k / a; e1 = np.array([1.0, 0, 0])
    par = np.dot(n, e1) * n; perp = e1 - par
    r = par * L + perp * np.sin(a * L) / a + np.cross(n, perp) * (1 - np.cos(a * L)) / a
    return r - L * e1

def reissner(F, L, EI, ES=np.inf, GS=np.inf):
    """planar cantilever, dead tip load F downward, extensible and shear deformable (Reissner); ES = GS = inf: the elastica.
    Shooting on the tip abscissa X; returns the tip (x, downward w) and the tip angle. Needs scipy."""
    from scipy.integrate import solve_ivp
    from scipy.optimize import brentq
    def run(X):
        def f(s, y):
            x, z, p = y
            e, g = F * np.sin(p) / ES, -F * np.cos(p) / GS
            return [(1 + e) * np.cos(p) + g * np.sin(p), -(1 + e) * np.sin(p) + g * np.cos(p), F * (X - x) / EI]
        return solve_ivp(f, [0, L], [0, 0, 0], rtol=1e-12, atol=1e-13).y[:, -1]
    X = brentq(lambda X: run(X)[0] - X, 0, 1.01 * L)
    x, z, p = run(X)
    return x, -z, p

## 2 · The Warren truss  *(lecture §A1–A3)*
The source's truss: 13 joints, 23 members, pinned at $(0,0)$, roller at $(6,0)$. One cell per member. Compare with the source (max deflection $-2.1882$ mm, horizontal 0.8333 mm) and
with the method of joints, which needs no $ES$ at all (the truss is statically determinate: $2j=m+r$, $26=23+3$).

In [ ]:
%%fenicsx -np 1
import sys; sys.path.insert(0, "/tmp/fnx")
from frame import *
# Warren truss of the source: span 6 m, height 1 m, 6 panels; bottom joints 0-6, top joints 7-12; 10 kN down at bottom joints 1-5
X = [(float(i), 0.0) for i in range(7)] + [(i + 0.5, 1.0) for i in range(6)]
E_ = [(i, i + 1) for i in range(6)] + [(7 + i, 8 + i) for i in range(5)] + sum([[(i, 7 + i), (i + 1, 7 + i)] for i in range(6)], [])
ES = 210e9 * 10e-4                                                 # E = 210 GPa, S = 10 cm^2

tr = Truss(X, E_, ES)
tr.fix((0, 0)); tr.fix((6, 0), [1])
for xl in range(1, 6):
    tr.load((xl, 0), (0, -10e3))
reason = tr.solve(); N = tr.forces(); u = tr.u.x.array.reshape(-1, 2)
say(f"reason {reason}: min u_y = {u[:, 1].min() * 1e3:.4f} mm, max |u_x| = {np.abs(u[:, 0]).max() * 1e3:.4f} mm")
# method of joints: equilibrium of every joint, unknowns = 23 member forces + 3 reactions
J = np.array(X); B = np.zeros((26, 26)); f = np.zeros(26)
for k, (i, j) in enumerate(E_):
    e = (J[j] - J[i]) / np.linalg.norm(J[j] - J[i])
    B[2 * i:2 * i + 2, k] += e; B[2 * j:2 * j + 2, k] -= e        # a tension N pulls joint i towards j
B[0, 23] = B[1, 24] = B[13, 25] = 1.0                              # reactions R_x(0), R_y(0), R_y(6)
for xl in range(1, 6):
    f[2 * xl + 1] = -10e3
Nj = np.linalg.solve(B, -f)
say(f"max |N_FEM - N_joints| = {np.abs(N - Nj[:23]).max():.2e} N;  reactions {Nj[23:] / 1e3} kN")
say("member forces [kN]: bottom chord " + " ".join(f"{x:.2f}" for x in N[:6] / 1e3) + " | top chord " + " ".join(f"{x:.2f}" for x in N[6:11] / 1e3)
    + " | diagonals " + " ".join(f"{x:.2f}" for x in N[11:] / 1e3))
tr2 = Truss(X, E_, ES, n=2); tr2.fix((0, 0)); tr2.fix((6, 0), [1])
for xl in range(1, 6):
    tr2.load((xl, 0), (0, -10e3))
say(f"two cells per member: KSP reason {tr2.solve()}, max |u| = {np.abs(tr2.u.x.array).max()}")

The same truss with two MPI ranks:

In [ ]:
%%fenicsx -np 2
import sys; sys.path.insert(0, "/tmp/fnx")
from frame import *
# Warren truss of the source: span 6 m, height 1 m, 6 panels; bottom joints 0-6, top joints 7-12; 10 kN down at bottom joints 1-5
X = [(float(i), 0.0) for i in range(7)] + [(i + 0.5, 1.0) for i in range(6)]
E_ = [(i, i + 1) for i in range(6)] + [(7 + i, 8 + i) for i in range(5)] + sum([[(i, 7 + i), (i + 1, 7 + i)] for i in range(6)], [])
ES = 210e9 * 10e-4                                                 # E = 210 GPa, S = 10 cm^2

tr = Truss(X, E_, ES)
tr.fix((0, 0)); tr.fix((6, 0), [1])
for xl in range(1, 6):
    tr.load((xl, 0), (0, -10e3))
reason = tr.solve()
umax = comm.allreduce(float(np.abs(tr.u.x.array).max()) if tr.u.x.array.size else 0.0, op=MPI.MAX)
say(f"-np {comm.size}: {tr.msh.topology.index_map(0).size_global} vertices (13 joints), KSP reason {reason}, max |u| = {umax}")

> **✏️ Your turn (5 min).** (i) Why do FEM and the method of joints agree to round-off, and why would they not for a truss with one extra member? (ii) What goes wrong with two cells
> per member? (iii) What happened with two ranks?

> *your answer*

## 3 · A Timoshenko cantilever: shear locking  *(lecture §B1–B3)*
The source's beam: $L=10$ m, $b\times h=0.3\times0.5$ m, $E=210$ GPa, $\nu=0.3$, $\kappa=5/6$, tip load $F_z=-1$ kN. Exact Timoshenko tip deflection $FL^3/(3EI)+FL/(\kappa GA)$. Three elements:
P1/P1 with full integration, P1/P1 with one-point integration of the shear terms, P2 displacement / P1 rotation.

In [ ]:
%%fenicsx -np 1
import sys; sys.path.insert(0, "/tmp/fnx")
from frame import *

L, b, h, E, nu, F = 10.0, 0.3, 0.5, 210e9, 0.3, -1e3
sec = rect(b, h, E, nu)                                  # beam along x, up = e_z: a1 = e_y, a2 = e_z, I1 = b h^3/12
EB = F * L**3 / (3 * sec["EI1"]); Tim = EB + F * L / sec["kGS2"]
say(f"Euler-Bernoulli {EB * 1e3:.4f} mm, Timoshenko {Tim * 1e3:.4f} mm (shear part {100 * (1 - EB / Tim):.2f} %)")
for elem in ("P1", "P1-RI", "P2P1"):
    row = []
    for n in (1, 2, 5, 10, 100):
        fr = Frame([(0, 0, 0), (L, 0, 0)], [(0, 1)], sec, n=n, elem=elem)
        fr.clamp((0, 0, 0)); fr.load((L, 0, 0), F=(0, 0, F)); fr.solve()
        row.append(f"n={n}: {fr.at((L, 0, 0))[0][2] / Tim:.6f}")
    say(f"{elem:6s} tip / Timoshenko: " + "  ".join(row))
say("(EB (1 - 1/(4 n^2)) + shear) / Timoshenko: " + "  ".join(f"n={n}: {(EB * (1 - 1 / (4 * n**2)) + (Tim - EB)) / Tim:.6f}" for n in (1, 2, 5, 10, 100)))

> **✏️ Your turn (5 min).** The source reports $-0.5025$ mm at $n=100$ as "1.07 % vs Euler–Bernoulli ✅". Is the element locking-free at $n=100$? Which number should it have been compared with?
> What does the last line tell you about the two locking-free elements?

> *your answer*

## 4 · Frames: bending with torsion, and the local frame  *(lecture §B2, §B4)*
**L-frame.** Leg 1 from $(0,0,0)$ to $(a,0,0)$, leg 2 to $(a,b,0)$, clamped at the origin, $F_z=-1$ kN at the free end; solid circle $r=0.05$ m. Exact (Timoshenko):
$w=\frac{F(a^3+b^3)}{3EI}+\frac{Fab^2}{GJ}+\frac{F(a+b)}{\kappa GA}$ — the middle term is the **twist** of leg 1 under the torque $Fb$.

In [ ]:
%%fenicsx -np 1
import sys; sys.path.insert(0, "/tmp/fnx")
from frame import *

a, b_, r, E, nu, F = 2.0, 1.0, 0.05, 210e9, 0.3, -1e3
s = circ(r, E, nu)
parts = F * (a**3 + b_**3) / (3 * s["EI1"]), F * a * b_**2 / s["GJ"], F * (a + b_) / s["kGS1"]
say("exact tip w = %.6e m  (bending %.4e, torsion %.4e, shear %.4e)" % ((sum(parts),) + parts))
for n in (4, 16):
    fr = Frame([(0, 0, 0), (a, 0, 0), (a, b_, 0)], [(0, 1), (1, 2)], s, n=n)
    fr.clamp((0, 0, 0)); fr.load((a, b_, 0), F=(0, 0, F)); fr.solve()
    R = fr.resultants(); leg1 = R["x"][:, 1] < 1e-9
    say(f"n = {n:2d}: tip w = {fr.at((a, b_, 0))[0][2]:.6e} (ratio {fr.at((a, b_, 0))[0][2] / sum(parts):.6f}),  twist of leg 1 at the corner {fr.at((a, 0, 0))[1][0]:.6e} "
        f"(F b a / GJ = {F * b_ * a / s['GJ']:.6e}),  torque in leg 1: {R['T'][leg1].min():.2f} .. {R['T'][leg1].max():.2f} N m")

**Portal frame.** Two columns of height $h=3$ m on fixed bases, a beam of span $\ell=4$ m, the same circular section, a horizontal load $F=1$ kN at the top of the left column. Slope–deflection
(axially rigid, no shear): $\Delta=\frac{Fh^3(2+3k)}{12EI(1+6k)}$, $k=\frac{I_b/\ell}{I_c/h}$. First with the default reference vector `up = e_z`.

In [ ]:
%%fenicsx -np 1
import sys; sys.path.insert(0, "/tmp/fnx")
from frame import *

h, l, r, E, nu, F = 3.0, 4.0, 0.05, 210e9, 0.3, 1e3
s = circ(r, E, nu); k = h / l
ref = F * h**3 * (2 + 3 * k) / (12 * s["EI1"] * (1 + 6 * k))
X = [(0, 0, 0), (0, 0, h), (l, 0, h), (l, 0, 0)]; M_ = [(0, 1), (1, 2), (2, 3)]
for up in ((0, 0, 1), (0, 1, 0)):
    for n in (8, 64):
        fr = Frame(X, M_, s, n=n, up=up)
        fr.clamp((0, 0, 0)); fr.clamp((l, 0, 0)); fr.load((0, 0, h), F=(F, 0, 0)); reason = fr.solve()
        d = fr.at((0, 0, h))[0][0]
        say(f"up = {up}, n = {n:2d}: KSP reason {reason:3d},  sway {d:.6e} m,  / slope-deflection {d / ref:.5f}")
rigid = dict(s, ES=1e6 * s["ES"], kGS1=1e6 * s["kGS1"], kGS2=1e6 * s["kGS2"])       # axially and shear rigid, as slope-deflection assumes
fr = Frame(X, M_, rigid, n=64, up=(0, 1, 0)); fr.clamp((0, 0, 0)); fr.clamp((l, 0, 0)); fr.load((0, 0, h), F=(F, 0, 0)); fr.solve()
say(f"axially and shear rigid, n = 64: / slope-deflection {fr.at((0, 0, h))[0][0] / ref:.5f}")

> **✏️ Your turn (5 min).** (i) Which term of the L-frame dominates, and why is the twist exact for every $n$? (ii) Why does `up = e_z` fail for the portal frame, and what would you do for a
> 3D frame with members in all directions? (iii) Why is the $n=8$ sway 1.5 % below slope–deflection, and the $n=64$ one 0.2 % above?

> *your answer*

## 5 · Finite rotations with exact answers: the circle and the helix  *(lecture §C1–C3)*
**Circle.** A tip moment $M$ about $-\mathbf e_y$ bends the rod into an arc of curvature $M/EI$ — with no force there is no axial or shear strain, so the arc is exact for any $ES$, $GS$.
$M=2\pi EI/L$ closes a full circle. **Helix.** For $EI=GJ$ a dead end moment $\mathbf m$ gives the constant spatial curvature $\mathbf m/EI$: $\boldsymbol\theta(s)=s\,\mathbf m/EI$ exactly and
the centre line is a helix about $\mathbf m$. Here $|\mathbf m|L/EI=3\pi$ (1.5 turns). Two ranks: the rod is a chain, parallel is fine.

In [ ]:
%%fenicsx -np 2
import sys; sys.path.insert(0, "/tmp/fnx")
from rod import *
L, EI, GJ, ES, GS = 10.0, 100.0, 100.0, 1e4, 1e4            # the source's rod (SI); ES L^2/EI = 1e4

rod = Rod(L, 40, ES, GS, GJ, EI)                                   # u in P2, theta in P1, two-point rule for the force term
for c in (0.25, 0.5, 0.75, 1.0, 1.25):
    M = c * 2 * np.pi * EI / L; res = rod.solve(M=(0, -M, 0)); u, th = rod.tip()
    say(f"M = {c:.2f} x 2 pi EI/L: SNES {res},  tip {np.round(u, 5)},  |error| {np.linalg.norm(u - circle_tip(L, EI, M)):.2e},  theta_y / exact {th[1] / (-M * L / EI):.8f}")
m = np.array([0.6, 0.0, 0.8]) * 3 * np.pi * EI / L
say(f"helix: exact tip {np.round(helix_tip(L, EI, m), 6)}")
for elem in ("P1-RI", "P2P1-RI"):
    for n in (20, 40):
        rod = Rod(L, n, ES, GS, GJ, EI, elem=elem); its = 0
        for c in np.linspace(0.1, 1.0, 10):
            res = rod.solve(M=c * m); its += res[1]
        u, th = rod.tip()
        say(f"{elem:7s} n = {n}: last SNES {res}, {its} Newton steps in 10 load steps;  |u - exact| {np.linalg.norm(u - helix_tip(L, EI, m)):.3e},  |theta - exact| {np.linalg.norm(th - L * m / EI):.1e}")

> **✏️ Your turn (5 min).** (i) The circle at $c=1.25$ goes beyond a full turn; why is that no problem for the rotation vector here? (ii) Why is $\boldsymbol\theta$ of the helix exact to round-off with
> both elements, while $\mathbf u$ is not? What are the convergence rates of $\mathbf u$?

> *your answer*

## 6 · A large-deflection cantilever and the elastica  *(lecture §C2, §C4)*
Dead tip load $F=\alpha EI/L^2$ downward, $\alpha$ up to 10 in 20 load steps. Reference: `reissner(F, L, EI, ES, GS)` — the planar Reissner rod (extensible, shear deformable) by
shooting; with $ES=GS=\infty$ it is the classical elastica. Compare the locking-free element with P1/P1 at full integration.

In [ ]:
%%fenicsx -np 2
import sys; sys.path.insert(0, "/tmp/fnx")
from rod import *
L, EI, GJ, ES, GS = 10.0, 100.0, 100.0, 1e4, 1e4            # the source's rod (SI); ES L^2/EI = 1e4

for elem in ("P2P1-RI", "P1"):
    rod = Rod(L, 40, ES, GS, GJ, EI, elem=elem); out = []
    for a in np.linspace(0.5, 10, 20):
        res = rod.solve(F=(0, 0, -a * EI / L**2))
        if a in (1.0, 2.0, 5.0, 10.0):
            u, th = rod.tip(); x, w, p = reissner(a * EI / L**2, L, EI, ES, GS); xe, we, pe = reissner(a * EI / L**2, L, EI)
            out.append(f"a = {a:4.1f}: w/L {-u[2] / L:.5f} (Reissner {w / L:.5f}, elastica {we / L:.5f}), 1 - x/L {-u[0] / L:.5f} ({1 - x / L:.5f})")
    say(f"{elem}, n = 40, last SNES {res}:\n  " + "\n  ".join(out))

> **✏️ Your turn (5 min).** (i) The FEM differs from the elastica by about $10^{-4}\alpha$ in $w/L$. Physics or error? (Hint: with $ES=GS$, what does $\varepsilon\,\mathbf d_1+\gamma\,\mathbf d_2$ add up to?)
> (ii) Linear theory gives $w/L=\alpha/3$. At $\alpha=10$ the rod deflects 0.81 $L$, not 3.3 $L$: why? (iii) What happened to P1/P1?

> *your answer*

## 7 · LLM exercise: a space truss  *(lecture §D1–D2)*
A tripod: feet at radius $a=2$ m on $z=0$ (angles 90°, 210°, 330°), pinned; apex at $(0,0,3)$; load $\mathbf P=(5,0,-20)$ kN at the apex; $ES=2\times10^8$ N. Ask an LLM for a DOLFINx 0.11 script (a
1D mesh in 3D, axial strain from the cell tangent) that prints the three member forces and the apex displacement, **and** an independent check. Paste it below, run it, and judge it.
The reference cell after it uses `frame.py` and the equilibrium of the apex.

In [ ]:
%%fenicsx -np 1
# Paste the LLM's script here (keep the %%fenicsx line). Then run the reference cell below.

In [ ]:
%%fenicsx -np 1
import sys; sys.path.insert(0, "/tmp/fnx")
from frame import *

a, H, P, ES = 2.0, 3.0, np.array([5e3, 0.0, -20e3]), 2e8
feet = [(a * np.cos(np.radians(t)), a * np.sin(np.radians(t)), 0.0) for t in (90, 210, 330)]
X = [(0.0, 0.0, H)] + feet; M_ = [(0, 1), (0, 2), (0, 3)]
tr = Truss(X, M_, ES)
for p in feet:
    tr.fix(p)
tr.load(X[0], P); reason = tr.solve(); N = tr.forces()
E3 = np.array([(np.array(p) - np.array(X[0])) / np.linalg.norm(np.array(p) - np.array(X[0])) for p in feet]).T   # columns: apex -> foot
Nref = np.linalg.solve(E3, -P)                                     # apex equilibrium: sum N_k e_k + P = 0
ua = tr.u.x.array.reshape(-1, 3)[vertex_dofs(tr.V, X[0])[0]]          # V is blocked: vertex_dofs gives the node (block) index
lk = np.linalg.norm(np.array(feet[0]) - np.array(X[0]))
uref = np.linalg.solve(E3.T, -Nref * lk / ES)                      # elongation of member k: -e_k . u_apex = N_k l / ES
say(f"KSP reason {reason};  N = {np.round(N / 1e3, 4)} kN,  equilibrium {np.round(Nref / 1e3, 4)} kN")
say(f"apex u = {ua} m,  from the elongations {uref} m")

> *your verdict on the LLM's code*